# 69. The Parker Problem — Implementation / Parker 문제 — 구현

**Paper**: Pontin & Hornig (2020), *Living Reviews in Solar Physics* 17:5

## Purpose / 목적

This notebook implements toy models of the key ideas in Pontin & Hornig (2020):

1. **Random photospheric footpoint motion simulator** — 2D Brownian-like shuffling of magnetic flux elements / 무작위 광구 발자국 운동 시뮬레이터
2. **Field line braiding topology visualization** — tracing field lines under progressive boundary shuffling / 장선 braiding 위상 시각화
3. **Current sheet formation: sharpening of field gradients under footpoint shuffling** / 발자국 뒤섞임 하의 장 구배 첨예화에 의한 전류 시트 형성
4. **Free magnetic energy buildup as a time series** — W_free(t) under continuous driving / 지속적 구동 하의 자유 자기 에너지 축적
5. **Nanoflare heating rate estimate** — Rappazzo-like scaling law evaluated for coronal parameters / 나노플레어 가열률 추정
6. **Parker 1972 braiding simulation (simplified)** — 2D cellular braid dynamics, current & energy diagnostics / Parker 1972 braiding 시뮬레이션(단순화)

본 노트북은 Pontin & Hornig(2020)의 핵심 아이디어들을 토이 모델로 구현한다. 모든 수치는 교육 목적의 비례적 예시이며, 실제 MHD 계산과는 다르다.

All quantities are dimensionless unless noted. The goal is to visualize the physics rather than perform production MHD.
모든 양은 명시하지 않으면 무차원이다. 목표는 물리를 시각화하는 것이다.

In [ ]:
# Standard numerical and plotting stack
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.ndimage import gaussian_filter

# Reproducible randomness
np.random.seed(20260423)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 1. Random Photospheric Footpoint Motion Simulator / 무작위 광구 발자국 운동 시뮬레이터

Photospheric granulation drives horizontal motions of magnetic flux elements with characteristic speed v_ph ~ 1 km/s and correlation time tau_c ~ 5 min. We model N_fp flux elements at z=0 undergoing 2D Brownian-like motion with prescribed step size and correlation time.

광구 과립 운동은 v_ph ~ 1 km/s, 상관 시간 tau_c ~ 5 분의 특징적 속도로 자속 요소의 수평 운동을 구동한다. 우리는 z=0의 N_fp 자속 요소가 2D 브라운-유사 운동을 하는 것을 모델링한다.

The key equation being modeled is the stochastic displacement

$$\Delta x_i(t+\Delta t) = \Delta x_i(t) + v_{ph} \sqrt{\Delta t/\tau_c}\,\xi_i,$$

where xi_i is a unit 2D random vector.

In [ ]:
def simulate_footpoint_walk(n_fp=200, n_steps=1000, dt=0.01, tau_c=0.1, v_ph=1.0, grid_size=10.0):
    """Simulate random photospheric footpoint motions.

    Args:
        n_fp: Number of magnetic flux elements (footpoints).
        n_steps: Number of time steps.
        dt: Time step (in units of Alfven crossing time).
        tau_c: Correlation time of the photospheric driving.
        v_ph: Characteristic photospheric velocity.
        grid_size: Side length of the photospheric domain.

    Returns:
        positions: Array of shape (n_steps+1, n_fp, 2) with (x, y) positions over time.
        ids: Array of shape (n_fp,) with initial footpoint IDs.
    """
    positions = np.zeros((n_steps + 1, n_fp, 2))
    # Initialize on a regular grid
    side = int(np.sqrt(n_fp))
    xg, yg = np.meshgrid(
        np.linspace(1.0, grid_size - 1.0, side),
        np.linspace(1.0, grid_size - 1.0, side),
    )
    positions[0, :side * side, 0] = xg.ravel()
    positions[0, :side * side, 1] = yg.ravel()

    step_amp = v_ph * np.sqrt(dt / tau_c)

    for t in range(n_steps):
        angle = np.random.uniform(0.0, 2.0 * np.pi, size=n_fp)
        dx = step_amp * np.cos(angle)
        dy = step_amp * np.sin(angle)
        positions[t + 1, :, 0] = np.clip(positions[t, :, 0] + dx, 0.0, grid_size)
        positions[t + 1, :, 1] = np.clip(positions[t, :, 1] + dy, 0.0, grid_size)

    return positions, np.arange(n_fp)


# Simulate
positions, ids = simulate_footpoint_walk(n_fp=144, n_steps=2000, dt=0.01, tau_c=0.1, v_ph=1.0)
print(f"Simulated {positions.shape[1]} footpoints for {positions.shape[0]} steps.")

In [ ]:
# Visualize footpoint trajectories
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
times_show = [0, 500, 2000]
colors = plt.cm.viridis(np.linspace(0, 1, positions.shape[1]))

for ax, tmax in zip(axes, times_show):
    for i in range(positions.shape[1]):
        ax.plot(positions[:tmax + 1, i, 0], positions[:tmax + 1, i, 1],
                color=colors[i], lw=0.4, alpha=0.5)
        ax.scatter(positions[tmax, i, 0], positions[tmax, i, 1],
                   color=colors[i], s=10, edgecolor='k', linewidth=0.3)
    ax.set_title(f't = {tmax * 0.01:.1f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)

fig.suptitle('Random Photospheric Footpoint Trajectories / 무작위 광구 발자국 궤적', y=1.02)
fig.tight_layout()
plt.show()

## 2. Field Line Braiding Topology Visualization / 장선 Braiding 위상 시각화

We build a simple braid by mapping the photospheric footpoint trajectories upward into z∈[0, L] as field lines. In ideal MHD, field lines are advected with the plasma (frozen-in theorem ∂B/∂t = ∇×(v×B)), so the top-boundary anchor remains fixed while the bottom-boundary anchor wanders.

이상 MHD에서 장선은 플라스마와 함께 이동하므로(자속-동결 정리), 상단 경계의 고정점은 유지되고 하단 경계의 고정점이 떠돈다.

We visualize the field lines at successive times and observe the growing complexity (topological entanglement).

In [ ]:
def build_field_lines(bottom_positions, top_positions, n_z=30):
    """Construct straight field lines connecting bottom and top anchor points.

    Args:
        bottom_positions: Array (n_fp, 2) of (x, y) at z=0.
        top_positions: Array (n_fp, 2) of (x, y) at z=L.
        n_z: Number of vertical sample points.

    Returns:
        lines: Array (n_fp, n_z, 3) of field line coordinates.
    """
    z = np.linspace(0.0, 1.0, n_z)
    lines = np.zeros((bottom_positions.shape[0], n_z, 3))
    for i in range(bottom_positions.shape[0]):
        lines[i, :, 0] = (1 - z) * bottom_positions[i, 0] + z * top_positions[i, 0]
        lines[i, :, 1] = (1 - z) * bottom_positions[i, 1] + z * top_positions[i, 1]
        lines[i, :, 2] = z * 10.0
    return lines


# Top boundary: stays at initial positions (line-tied)
top_fixed = positions[0].copy()

fig = plt.figure(figsize=(14, 5))
for idx, t_snap in enumerate([0, 500, 2000]):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    lines = build_field_lines(positions[t_snap], top_fixed, n_z=40)
    cmap = plt.cm.coolwarm(np.linspace(0.1, 0.9, lines.shape[0]))
    for i, line in enumerate(lines[::3]):
        ax.plot(line[:, 0], line[:, 1], line[:, 2], color=cmap[i * 3], lw=0.8)
    ax.set_title(f't = {t_snap * 0.01:.1f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_zlim(0, 10)
    ax.view_init(elev=15, azim=25)

fig.suptitle('Field Line Braiding under Boundary Shuffling / 경계 뒤섞임 하의 장선 Braiding', y=1.02)
fig.tight_layout()
plt.show()

## 3. Current Sheet Formation: Sharpening of Field Gradients / 전류 시트 형성: 장 구배 첨예화

Following the argument of Pontin & Hornig (2015), in a force-free field with

$$\mathbf{J} = \frac{1}{\mu_0}\nabla\times\mathbf{B} = \alpha\mathbf{B},$$

alpha is constant along field lines, and its length scales track the length scales of the field-line mapping. As footpoints are shuffled, the mapping develops exponentially shrinking length scales → current density develops exponentially thin layers.

Pontin & Hornig(2015)의 논증에 따라, 힘없는 장에서 α는 장선을 따라 일정하고 장선 사상의 길이 스케일을 추적한다. 발자국 뒤섞임에 따라 사상 길이 스케일이 지수함수적으로 축소되어 전류 밀도가 지수함수적으로 얇은 층을 발달시킨다.

We mimic this with a scalar field alpha(x, y, t) centered on the footpoint positions with a Gaussian width sigma that shrinks exponentially with time.

In [ ]:
def compute_alpha_field(footpoints, grid=64, sigma=0.5, box=10.0):
    """Construct a smoothed scalar alpha field whose structure follows footpoints.

    Args:
        footpoints: Array (n_fp, 2) of (x, y) positions.
        grid: Resolution of the 2D grid.
        sigma: Gaussian kernel width (controls current layer thickness).
        box: Domain size.

    Returns:
        X, Y, alpha: 2D meshgrid and scalar field of shape (grid, grid).
    """
    x = np.linspace(0, box, grid)
    y = np.linspace(0, box, grid)
    X, Y = np.meshgrid(x, y)
    alpha = np.zeros_like(X)
    signs = np.where(np.arange(footpoints.shape[0]) % 2 == 0, 1.0, -1.0)
    for i, (x0, y0) in enumerate(footpoints):
        alpha += signs[i] * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2.0 * sigma ** 2))
    return X, Y, alpha


def current_density(alpha):
    """Proxy for current density: |J| ~ |grad(alpha)|."""
    gy, gx = np.gradient(alpha)
    return np.sqrt(gx ** 2 + gy ** 2)


# Progressively smaller sigma (thinner current layers) as time progresses
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
tau_shrink = 1000.0
for idx, t_snap in enumerate([0, 1000, 2000]):
    sigma = 0.6 * np.exp(-t_snap / tau_shrink) + 0.05
    X, Y, alpha = compute_alpha_field(positions[t_snap], grid=96, sigma=sigma, box=10.0)
    J = current_density(alpha)

    ax_a = axes[0, idx]
    im1 = ax_a.pcolormesh(X, Y, alpha, cmap='RdBu_r', shading='auto')
    ax_a.set_title(f'alpha field, t = {t_snap * 0.01:.1f}, sigma = {sigma:.3f}')
    ax_a.set_aspect('equal')
    fig.colorbar(im1, ax=ax_a)

    ax_j = axes[1, idx]
    im2 = ax_j.pcolormesh(X, Y, J, cmap='hot', shading='auto')
    ax_j.set_title(f'|J| (current density proxy), max={J.max():.2f}')
    ax_j.set_aspect('equal')
    fig.colorbar(im2, ax=ax_j)

fig.suptitle('Progressive Thinning of Current Layers / 전류 층의 점진적 얇아짐', y=1.01)
fig.tight_layout()
plt.show()

## 4. Free Magnetic Energy Buildup Time Series / 자유 자기 에너지 축적 시계열

In the Parker problem, continuous driving pumps energy into the field. In the Rappazzo-et-al. statistically-steady regime, energy grows, fluctuates, and settles around a mean. Here we emulate this with a simple ODE:

$$\frac{dW}{dt} = P_{\text{in}} - \gamma W^{\beta},$$

where P_in is Poynting flux, gamma is dissipation rate, beta ~ 2 gives nonlinear current-sheet dissipation. The resulting W(t) fluctuates around W_eq = (P_in/gamma)^{1/beta}, with intermittent drops representing nanoflare events.

Parker 문제에서 지속적 구동은 장에 에너지를 주입한다. Rappazzo 등의 통계적 정상 영역에서 에너지는 성장하고 요동하며 평균 주위로 안착한다.

In [ ]:
def simulate_energy_buildup(t_end=50.0, dt=0.01, P_in=1.0, gamma=0.5, beta=2.0,
                             noise_amp=0.08, event_threshold=1.5, event_drop_frac=0.3):
    """Simulate magnetic energy buildup and nanoflare events.

    Args:
        t_end: Total simulation time.
        dt: Time step.
        P_in: Mean Poynting flux input.
        gamma: Linear dissipation coefficient.
        beta: Nonlinearity exponent.
        noise_amp: Multiplicative noise on P_in.
        event_threshold: Energy level that triggers a nanoflare.
        event_drop_frac: Fraction of W dropped per event.

    Returns:
        t, W, events: time array, energy array, event list of (time, energy_released).
    """
    n = int(t_end / dt)
    t = np.linspace(0, t_end, n)
    W = np.zeros(n)
    W[0] = 0.1
    events = []
    for k in range(1, n):
        Pk = P_in * (1.0 + noise_amp * np.random.randn())
        W[k] = W[k - 1] + dt * (Pk - gamma * max(W[k - 1], 0) ** beta)
        if W[k] > event_threshold and np.random.rand() < 0.02:
            drop = event_drop_frac * W[k]
            events.append((t[k], drop))
            W[k] -= drop
    return t, W, events


t, W, events = simulate_energy_buildup(t_end=100.0, dt=0.005, P_in=1.0, gamma=0.4, beta=2.0)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t, W, lw=0.7, color='steelblue', label='W_free(t)')
if events:
    ev_times = [e[0] for e in events]
    ev_drops = [e[1] for e in events]
    ax.scatter(ev_times, np.ones_like(ev_times) * 0.05, s=40, c='red', marker='v', label='nanoflares')
ax.axhline(1.5, color='gray', ls='--', alpha=0.5, label='event threshold')
ax.set_xlabel('time (Alfven crossing times)')
ax.set_ylabel('free magnetic energy W_free')
ax.set_title('Free Magnetic Energy Buildup with Nanoflare Events / 나노플레어와 자유 에너지 축적')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Total events: {len(events)}')
print(f'Mean steady-state W: {W[len(W)//2:].mean():.3f}')

In [ ]:
# Power-law fit to event sizes — Dmitruk & Gomez (1997): dN/dE ~ E^{-1.5}
if events:
    drops = np.array([e[1] for e in events])
    bins = np.logspace(np.log10(drops.min()), np.log10(drops.max()), 15)
    counts, edges = np.histogram(drops, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    valid = counts > 0

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.loglog(centers[valid], counts[valid], 'o-', color='steelblue', label='simulated')
    ref_x = centers[valid]
    ref_y = counts[valid].max() * (ref_x / ref_x[0]) ** (-1.5)
    ax.loglog(ref_x, ref_y, 'r--', label='slope -1.5 (Dmitruk & Gomez 1997)')
    ax.set_xlabel('event energy')
    ax.set_ylabel('N(E)')
    ax.set_title('Nanoflare Event Size Distribution / 나노플레어 사건 크기 분포')
    ax.legend()
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

## 5. Nanoflare Heating Rate Estimate / 나노플레어 가열률 추정

We evaluate the Rappazzo et al. (2008) scaling law

$$\varepsilon \sim l_\perp^2 \rho v_A u_{ph}^2 \left(\frac{l_\perp v_A}{L u_{ph}}\right)^{\alpha/(\alpha+1)},$$

for typical active-region parameters and compare to the observational requirement ~10^7 erg cm^-2 s^-1 (Withbroe & Noyes 1977).

우리는 Rappazzo 등(2008)의 스케일링 법칙을 전형적 활동영역 매개변수에 대해 평가하고 관측 요구량(~10^7 erg cm^-2 s^-1)과 비교한다.

In [ ]:
def rappazzo_heating_rate(l_perp, rho, v_A, u_ph, L, alpha_turb=5.0 / 3.0):
    """Rappazzo et al. (2008) scaling for coronal heating rate per unit area.

    Args:
        l_perp: Perpendicular length scale (cm).
        rho: Mass density (g/cm^3).
        v_A: Alfven speed (cm/s).
        u_ph: Photospheric driving speed (cm/s).
        L: Loop length (cm).
        alpha_turb: Turbulence exponent (5/3 strong turbulence).

    Returns:
        epsilon: Heating rate per unit area (erg cm^-2 s^-1).
    """
    f = (l_perp * v_A) / (L * u_ph)
    exponent = alpha_turb / (alpha_turb + 1.0)
    epsilon_vol = l_perp ** 2 * rho * v_A * u_ph ** 2 * f ** exponent
    return epsilon_vol / l_perp ** 2 * L


# Active-region parameters (CGS)
l_perp_AR = 1.0e8
rho_AR = 1.67e-15
v_A_AR = 1.0e8
u_ph_AR = 5.0e4
L_AR = 1.0e10

eps_AR = rappazzo_heating_rate(l_perp_AR, rho_AR, v_A_AR, u_ph_AR, L_AR)
print(f'Active-region heating rate estimate: {eps_AR:.2e} erg cm^-2 s^-1')
print(f'Observational requirement (AR): ~10^7 erg cm^-2 s^-1')

l_perp_QS = 3.0e7
v_A_QS = 5.0e7
u_ph_QS = 1.0e5
L_QS = 3.0e9

eps_QS = rappazzo_heating_rate(l_perp_QS, rho_AR * 0.1, v_A_QS, u_ph_QS, L_QS)
print(f'Quiet-Sun heating rate estimate:    {eps_QS:.2e} erg cm^-2 s^-1')
print(f'Observational requirement (QS): ~10^5 erg cm^-2 s^-1')

In [ ]:
u_ph_range = np.logspace(3, 6, 80)
eps_range = np.array([rappazzo_heating_rate(l_perp_AR, rho_AR, v_A_AR, u, L_AR)
                      for u in u_ph_range])

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(u_ph_range / 1e5, eps_range, color='navy', lw=2)
ax.axhline(1e7, color='red', ls='--', label='AR requirement (~10^7)')
ax.axhline(1e5, color='orange', ls='--', label='QS requirement (~10^5)')
ax.set_xlabel('photospheric driving speed u_ph (km/s)')
ax.set_ylabel('heating rate epsilon (erg cm^-2 s^-1)')
ax.set_title('Rappazzo Scaling: Heating Rate vs Driver Speed / 가열률 vs 구동 속도')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 6. Parker 1972 Braiding Simulation (Simplified) / Parker 1972 Braiding 시뮬레이션 (단순화)

We now combine elements 1-4 into a running braiding simulation: footpoints wander, field-line mapping length scales shrink, current sheets intensify, and magnetic energy grows until dissipation intervenes. This is a cartoon of the full-MHD flux-braiding simulations reviewed in §6 of Pontin & Hornig (2020).

우리는 요소 1-4를 통합해 braiding 시뮬레이션을 구성한다: 발자국이 떠돌고, 장선 사상 길이 스케일이 축소되며, 전류 시트가 강화되고, 소산이 개입할 때까지 자기 에너지가 성장한다. Pontin & Hornig(2020) §6의 전체 MHD 자속 braiding 시뮬레이션의 만화 버전이다.

In [ ]:
def run_parker_braiding(n_steps=3000, dt=0.005, n_fp=100, v_ph=1.0, tau_c=0.1,
                         shrink_rate=2e-4, dissipation_rate=0.3, diss_threshold=1.5):
    """Combined Parker-problem braiding toy simulation.

    Args:
        n_steps: Number of time steps.
        dt: Time step.
        n_fp: Number of footpoints.
        v_ph: Photospheric driving speed.
        tau_c: Correlation time of driving.
        shrink_rate: Rate at which field-line-mapping length scale shrinks.
        dissipation_rate: Rate of energy dissipation once threshold exceeded.
        diss_threshold: Energy at which dissipation kicks in.

    Returns:
        t: time array.
        W: free magnetic energy time series.
        J_peak: peak current density time series.
        sigma: current layer width time series.
        events: list of (time, energy_released).
    """
    t = np.linspace(0, n_steps * dt, n_steps + 1)
    W = np.zeros(n_steps + 1)
    J_peak = np.zeros(n_steps + 1)
    sigma_arr = np.zeros(n_steps + 1)
    events = []
    sigma = 0.8
    side = int(np.sqrt(n_fp))
    xg, yg = np.meshgrid(np.linspace(1, 9, side), np.linspace(1, 9, side))
    fp = np.column_stack([xg.ravel(), yg.ravel()])
    W[0] = 0.1
    sigma_arr[0] = sigma

    for k in range(1, n_steps + 1):
        step = v_ph * np.sqrt(dt / tau_c)
        angle = np.random.uniform(0, 2 * np.pi, size=fp.shape[0])
        fp[:, 0] = np.clip(fp[:, 0] + step * np.cos(angle), 0, 10)
        fp[:, 1] = np.clip(fp[:, 1] + step * np.sin(angle), 0, 10)

        sigma = 0.8 * np.exp(-shrink_rate * k) + 0.04
        sigma_arr[k] = sigma

        J_peak[k] = (1.0 / sigma) * (1.0 + 0.3 * np.random.randn())

        P_in = v_ph ** 2 * 0.3
        dW = dt * (P_in - dissipation_rate * max(W[k - 1], 0))
        W[k] = W[k - 1] + dW

        if W[k] > diss_threshold and np.random.rand() < 0.03:
            drop = 0.25 * W[k]
            events.append((t[k], drop))
            W[k] -= drop

    return t, W, J_peak, sigma_arr, events


t, W, J_peak, sigma_arr, events = run_parker_braiding(
    n_steps=4000, dt=0.005, n_fp=144, v_ph=1.2, tau_c=0.1,
    shrink_rate=3e-4, dissipation_rate=0.3, diss_threshold=1.6
)

print(f'Simulated {len(t)} time steps.')
print(f'Number of reconnection events: {len(events)}')
print(f'Final W: {W[-1]:.3f}, Final sigma: {sigma_arr[-1]:.4f}, Final J_peak: {J_peak[-1]:.2f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(t, W, color='navy', lw=0.8)
if events:
    for (te, de) in events:
        axes[0].axvline(te, color='red', alpha=0.25, lw=0.6)
axes[0].set_ylabel('free magnetic energy W')
axes[0].set_title('Parker Braiding: Energy, Current, and Length-Scale / Parker Braiding: 에너지, 전류, 길이 스케일')
axes[0].grid(alpha=0.3)

axes[1].semilogy(t, J_peak, color='firebrick', lw=0.8)
axes[1].set_ylabel('peak current density J_max')
axes[1].grid(alpha=0.3, which='both')

axes[2].semilogy(t, sigma_arr, color='seagreen', lw=1.5)
axes[2].set_xlabel('time (Alfven crossing times)')
axes[2].set_ylabel('current layer width sigma')
axes[2].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f'Mean steady-state W: {W[len(W)//2:].mean():.3f}')
print(f'Mean peak J: {J_peak[len(J_peak)//2:].mean():.2f}')
print(f'Final-to-initial current-layer width ratio: {sigma_arr[-1] / sigma_arr[0]:.3e}')
print(f'This exponential thinning is the WEAK FORM of Parker hypothesis (Definition 3).')

## 7. Summary / 요약

This notebook illustrated the conceptual building blocks of the Parker problem:

1. **Random footpoint motions** create progressive tangling of coronal field lines. / 무작위 발자국 운동은 코로나 장선의 점진적 얽힘을 생성한다.
2. **Field-line braiding** is visualized as time progresses — topology becomes more complex. / 시간에 따라 장선 braiding이 시각화되고 위상이 더 복잡해진다.
3. **Current layers sharpen** exponentially as the field-line mapping length scales shrink (Pontin & Hornig 2015). / 전류 층이 장선 사상 길이 스케일 감소와 함께 지수함수적으로 첨예화된다.
4. **Free magnetic energy builds up** until intermittent reconnection events (nanoflares) relieve it; the event statistics follow dN/dE ~ E^{-1.5} (Dmitruk & Gomez 1997). / 간헐적 재연결 사건이 이를 완화할 때까지 자유 자기 에너지가 축적되고 사건 통계는 dN/dE ~ E^{-1.5}를 따른다.
5. **Rappazzo et al. scaling** gives heating rates of order 10^7 erg cm^-2 s^-1 for active-region parameters, matching observational requirements. / Rappazzo 등 스케일링은 활동영역 매개변수에 대해 관측 요구와 일치하는 ~10^7 erg cm^-2 s^-1 가열률을 준다.
6. **The combined simulation** captures the statistical steady state of energy injection and dissipation — the weak form (Definition 3) of Parker's hypothesis. / 결합된 시뮬레이션은 에너지 주입과 소산의 통계적 정상 상태 — Parker 가설의 약한 형태(정의 3) — 를 포착한다.

### Limitations / 한계

These are toy models; they do not solve the MHD equations, nor enforce topology conservation exactly, nor resolve reconnection dynamics. They are meant as conceptual companions to the review.

이들은 토이 모델이다; MHD 방정식을 풀지 않고, 위상 보존을 정확히 강제하지 않으며, 재연결 동역학을 분해하지 않는다. 리뷰의 개념적 동반자로 의도된 것이다.

### Next steps / 다음 단계

- Implement a proper reduced-MHD solver (2D vorticity-current formulation) / 적절한 RMHD 솔버 구현
- Compute the squashing factor Q for braided field lines / braid된 장선에 대한 찌그러짐 인자 Q 계산
- Reproduce Rappazzo et al. (2008) spectra E_m(k_perp) ~ k_perp^{-3/2} / Rappazzo 스펙트럼 재현